In [ ]:
#text_gen_aug using aragpt2-base

import json
import copy
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Initialize Arabic text generation pipeline (AraGPT2)
model_name = "aubmindlab/aragpt2-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
device = 0 if torch.cuda.is_available() else -1
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

def augment_text(text, num_return_sequences=1):
    outputs = generator(
        text,
        max_new_tokens=10,  # Generate up to 10 new tokens
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_p=0.95,
        temperature=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    return outputs[0]['generated_text']

def replace_all_occurrences(text, old, new):
    return text.replace(old, new)

def augment_entity(entity, techniques_to_augment, num_augmentations=1):
    augmented_entities = []  # include original
    for technique in techniques_to_augment:
        target_labels = [label for label in entity.get("labels", []) if label["technique"] == technique]
        for label in target_labels:
            original_span = label["text"]
            for _ in range(num_augmentations):
                aug_span = augment_text(original_span)
                if aug_span.strip() == original_span.strip():
                    continue  # skip identical augmentations
                new_entity = copy.deepcopy(entity)
                new_entity["text"] = replace_all_occurrences(new_entity["text"], original_span, aug_span)
                for lbl in new_entity.get("labels", []):
                    if lbl["technique"] == technique and lbl["text"] == original_span:
                        lbl["text"] = aug_span
                        start_pos = new_entity["text"].find(aug_span)
                        if start_pos != -1:
                            lbl["start"] = start_pos
                            lbl["end"] = start_pos + len(aug_span)
                augmented_entities.append(new_entity)
    return augmented_entities

def augment_dataset_file(input_json_path, output_json_path, techniques_to_augment, num_augmentations=1):
    with open(input_json_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    
    augmented_dataset = []  # This will now ONLY contain augmented samples (no original data)
    
    for entity in dataset:
        if not entity.get("labels"):
            continue  # Skip entities without labels (since we only want augmented data)
        
        entity_techniques = {label["technique"] for label in entity["labels"]}
        common_techniques = entity_techniques.intersection(set(techniques_to_augment))
        if not common_techniques:
            continue  # Skip if no matching techniques (we don't want the original)
        
        augmented_entities = augment_entity(entity, techniques_to_augment, num_augmentations)
        augmented_dataset.extend(augmented_entities)  # Only augmented samples are added
    
    with open(output_json_path, 'w', encoding='utf-8') as f_out:
        json.dump(augmented_dataset, f_out, ensure_ascii=False, indent=2)

    print(f"Augmentation completed. Original dataset size: {len(dataset)}. Augmented-only dataset size: {len(augmented_dataset)}.")


if __name__ == "__main__":
    input_file = "/kaggle/input/enhanced_augmented_sub_data.json"     # synonum replacement augmentation using LLM(Cohere)
    output_file = "Doubt_text_generation4.json"  # Output path for augmented data
    #techniques = ["Exaggeration/Minimisation", "Appeal to authority", "Appeal to fear/prejudice",
    #"Black-and-white Fallacy/Dictatorship","Causal Oversimplification","Doubt", "Flag-waving",
    #"Glittering generalities (Virtue)","Obfuscation, Intentional vagueness, Confusion", 
    #"Presenting Irrelevant Data (Red Herring)", "Repetition", "Slogans", "Smears", 
    #"Thought-terminating cliché", "Whataboutism"]  # Techniques you want to augment
    techniques = ["Doubt"]  # Techniques to augmentaugment_per_entity = 10  # Number of augmentations per technique per entity
    augment_per_entity = 4 
    augment_dataset_file(input_file, output_file, techniques, augment_per_entity)


In [13]:
#Back-translation augmentation
import json
import copy
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# Setup device
device = 0 if torch.cuda.is_available() else -1

# Load translation pipelines for back-translation (Arabic <-> English)
ar_to_en = pipeline("translation", model="Helsinki-NLP/opus-mt-ar-en", device=device)
en_to_ar = pipeline("translation", model="Helsinki-NLP/opus-mt-en-ar", device=device)

def back_translate(text):
    # Translate Arabic -> English
    en_translation = ar_to_en(text, max_length=100, truncation=True)[0]['translation_text']
    # Translate English -> Arabic
    back_translated = en_to_ar(en_translation, max_length=100, truncation=True)[0]['translation_text']
    return back_translated

def replace_all_occurrences(text, old, new):
    return text.replace(old, new)

def augment_entity_backtranslation(entity, techniques_to_augment, num_augmentations=1):
    augmented_entities = []  
    for technique in techniques_to_augment:
        target_labels = [label for label in entity.get("labels", []) if label["technique"] == technique]
        for label in target_labels:
            original_span = label["text"]
            for _ in range(num_augmentations):
                bt_span = back_translate(original_span)
                if bt_span.strip() == original_span.strip():
                    continue  # Skip if no effective augmentation
                
                new_entity = copy.deepcopy(entity)
                # Replace all occurrences of original_span with back-translated span in outer text
                new_entity["text"] = replace_all_occurrences(new_entity["text"], original_span, bt_span)
                
                # Update label's text and span positions accordingly
                for lbl in new_entity.get("labels", []):
                    if lbl["technique"] == technique and lbl["text"] == original_span:
                        lbl["text"] = bt_span
                        start_pos = new_entity["text"].find(bt_span)
                        if start_pos != -1:
                            lbl["start"] = start_pos
                            lbl["end"] = start_pos + len(bt_span)
                augmented_entities.append(new_entity)
    return augmented_entities

def augment_dataset_backtranslation(input_json_path, output_json_path, techniques_to_augment, num_augmentations=1):
    with open(input_json_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    
    augmented_dataset = []
    for entity in dataset:
        if not entity.get("labels"):
            continue  # Skip entries with no labels (no augmentation and no original)
        
        entity_techniques = {label["technique"] for label in entity["labels"]}
        common_techniques = entity_techniques.intersection(set(techniques_to_augment))
        if not common_techniques:
            continue  # Skip entries with no augmentable techniques
        
        augmented_entities = augment_entity_backtranslation(entity, techniques_to_augment, num_augmentations)
        augmented_dataset.extend(augmented_entities)
    
    with open(output_json_path, 'w', encoding='utf-8') as f_out:
        json.dump(augmented_dataset, f_out, ensure_ascii=False, indent=2)

    print(f"Back-translation augmentation done. Original size: {len(dataset)}. Augmented dataset size: {len(augmented_dataset)}.")

# ===== Usage example =====
if __name__ == "__main__":
    input_file = "/kaggle/input/enhanced_augmented_sub_data.json"     # synonum replacement augmentation using LLM(Cohere)
    output_file = "What_trans_aug.json"  # Output path for augmented data only
    #techniques = ["Exaggeration/Minimisation", "Appeal to authority", "Appeal to fear/prejudice",
    #"Black-and-white Fallacy/Dictatorship","Causal Oversimplification","Doubt", "Flag-waving",
    #"Glittering generalities (Virtue)","Obfuscation, Intentional vagueness, Confusion", 
    #"Presenting Irrelevant Data (Red Herring)", "Repetition", "Slogans", "Smears", 
    #"Thought-terminating cliché", "Whataboutism"]  # Techniques you want to augment
    techniques = ["Whataboutism"]  # Techniques you want to augment
    augment_per_entity = 1  # Number of augmentations per technique per entity
    
    augment_dataset_backtranslation(input_file, output_file, techniques, augment_per_entity)


Device set to use cuda:0
Device set to use cuda:0


Back-translation augmentation done. Original size: 1194. Augmented dataset size: 7.
